In [6]:
!pip install transformers datasets torch scikit-learn pandas openpyxl matplotlib -q

In [10]:
from google.colab import files

uploaded = files.upload()

Saving iva_role4_dental_form_classifier_dataset.csv to iva_role4_dental_form_classifier_dataset.csv


In [11]:
import pandas as pd

df = pd.read_csv("iva_role4_dental_form_classifier_dataset.csv")

df.head()

,example_id,text,label,diagnosis,procedure,medication,tooth_number,source_type
0,46,Tooth 45 is not restorable due to non-restorab...,extraction_form,non-restorable tooth,surgical extraction,clindamycin,45.0,synthetic_dental_form_example
1,112,Patient reports gum bleeding. Exam shows mild ...,cleaning_form,mild periodontitis,deep cleaning,fluoride,NaN,synthetic_dental_form_example
2,116,Patient presents with bleeding gums. Recommend...,cleaning_form,bleeding gums,scaling and polishing,fluoride,NaN,synthetic_dental_form_example
3,113,Hygiene appointment completed. Findings includ...,cleaning_form,bleeding gums,dental cleaning,none,NaN,synthetic_dental_form_example
4,59,Oral surgery appointment: tooth extraction pla...,extraction_form,severe decay,tooth extraction,clindamycin,35.0,synthetic_dental_form_example


In [12]:
df.columns

Index(['example_id', 'text', 'label', 'diagnosis', 'procedure', 'medication',
       'tooth_number', 'source_type'],
      dtype='object')

In [13]:
df["label"].value_counts()

,count
label,
extraction_form,30
cleaning_form,30
filling_form,30
consultation_form,30
root_canal_form,30


In [14]:
labels = sorted(df["label"].unique())

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

df["label_id"] = df["label"].map(label2id)

print(label2id)
df.head()

{'cleaning_form': 0, 'consultation_form': 1, 'extraction_form': 2, 'filling_form': 3, 'root_canal_form': 4}


,example_id,text,label,diagnosis,procedure,medication,tooth_number,source_type,label_id
0,46,Tooth 45 is not restorable due to non-restorab...,extraction_form,non-restorable tooth,surgical extraction,clindamycin,45.0,synthetic_dental_form_example,2
1,112,Patient reports gum bleeding. Exam shows mild ...,cleaning_form,mild periodontitis,deep cleaning,fluoride,NaN,synthetic_dental_form_example,0
2,116,Patient presents with bleeding gums. Recommend...,cleaning_form,bleeding gums,scaling and polishing,fluoride,NaN,synthetic_dental_form_example,0
3,113,Hygiene appointment completed. Findings includ...,cleaning_form,bleeding gums,dental cleaning,none,NaN,synthetic_dental_form_example,0
4,59,Oral surgery appointment: tooth extraction pla...,extraction_form,severe decay,tooth extraction,clindamycin,35.0,synthetic_dental_form_example,2


In [15]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label_id"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label_id"],
    random_state=42
)

print(len(train_df), len(val_df), len(test_df))

120 15 15


In [16]:
from datasets import Dataset, DatasetDict

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df)
})

dataset

DatasetDict({
    train: Dataset({
        features: ['example_id', 'text', 'label', 'diagnosis', 'procedure', 'medication', 'tooth_number', 'source_type', 'label_id', '__index_level_0__'],
        num_rows: 120
    })
    validation: Dataset({
        features: ['example_id', 'text', 'label', 'diagnosis', 'procedure', 'medication', 'tooth_number', 'source_type', 'label_id', '__index_level_0__'],
        num_rows: 15
    })
    test: Dataset({
        features: ['example_id', 'text', 'label', 'diagnosis', 'procedure', 'medication', 'tooth_number', 'source_type', 'label_id', '__index_level_0__'],
        num_rows: 15
    })
})

In [17]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [18]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

tokenized_dataset = tokenized_dataset.rename_column("label_id", "labels")

tokenized_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

In [19]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels_true = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels_true, predictions),
        "f1": f1_score(labels_true, predictions, average="weighted")
    }

training_args = TrainingArguments(
    output_dir="./iva_form_classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_strategy="epoch",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.572755,1.511067,0.866667,0.862857
2,1.393412,1.347643,0.800000,0.801429
3,1.254116,1.271561,0.933333,0.931429


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=45, training_loss=1.406760533650716, metrics={'train_runtime': 337.1334, 'train_samples_per_second': 1.068, 'train_steps_per_second': 0.133, 'total_flos': 11922703718400.0, 'train_loss': 1.406760533650716, 'epoch': 3.0})

In [21]:
results = trainer.evaluate(tokenized_dataset["test"])
results

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 1.2662947177886963,
 'eval_accuracy': 0.9333333333333333,
 'eval_f1': 0.9314285714285714,
 'eval_runtime': 2.5716,
 'eval_samples_per_second': 5.833,
 'eval_steps_per_second': 0.778,
 'epoch': 3.0}

In [22]:
import torch

def predict_form(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=1)
    predicted_id = torch.argmax(probabilities, dim=1).item()
    confidence = probabilities[0][predicted_id].item()

    return {
        "form_type": id2label[predicted_id],
        "confidence": round(confidence, 3)
    }

In [23]:
predict_form("Patient has cavity on tooth 15 and needs composite filling.")

{'form_type': 'extraction_form', 'confidence': 0.28}

In [24]:
def autofill_form(text, diagnosis=None, procedure=None, medication=None, tooth_number=None):
    prediction = predict_form(text)

    form = {
        "form_type": prediction["form_type"],
        "confidence": prediction["confidence"],
        "diagnosis": diagnosis,
        "procedure": procedure,
        "medication": medication,
        "tooth_number": tooth_number,
        "original_text": text,
        "status": "ready_for_review"
    }

    return form

In [25]:
autofill_form(
    text="Patient has cavity on tooth 15 and needs composite filling.",
    diagnosis="cavity",
    procedure="composite filling",
    medication=None,
    tooth_number="15"
)

{'form_type': 'extraction_form',
 'confidence': 0.28,
 'diagnosis': 'cavity',
 'procedure': 'composite filling',
 'medication': None,
 'tooth_number': '15',
 'original_text': 'Patient has cavity on tooth 15 and needs composite filling.',
 'status': 'ready_for_review'}

In [26]:
results = trainer.evaluate(tokenized_dataset["test"])
results

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 1.2662947177886963,
 'eval_accuracy': 0.9333333333333333,
 'eval_f1': 0.9314285714285714,
 'eval_runtime': 2.754,
 'eval_samples_per_second': 5.447,
 'eval_steps_per_second': 0.726,
 'epoch': 3.0}

In [27]:
predict_form("Patient requires root canal treatment because of severe infection.")

{'form_type': 'cleaning_form', 'confidence': 0.241}

In [28]:
predict_form("Tooth 18 extraction is needed due to swelling and decay.")

{'form_type': 'extraction_form', 'confidence': 0.318}

In [29]:
predict_form("Patient has plaque buildup and bleeding gums, cleaning required.")

{'form_type': 'cleaning_form', 'confidence': 0.258}

In [30]:
#generate 1000 synthetic examples

In [31]:
import random
import pandas as pd

filling_templates = [
    "Patient has cavity on tooth {} and needs composite filling.",
    "Tooth {} requires filling due to decay.",
    "Dental examination showed cavity requiring filling on tooth {}.",
    "Patient reports sensitivity and filling is required on tooth {}."
]

extraction_templates = [
    "Tooth {} extraction is needed due to severe decay.",
    "Patient requires extraction of tooth {} because of infection.",
    "Swelling and pain indicate extraction is required for tooth {}.",
    "Tooth {} cannot be restored and must be extracted."
]

root_canal_templates = [
    "Patient requires root canal treatment on tooth {} because of infection.",
    "Severe nerve pain indicates root canal is needed for tooth {}.",
    "Root canal procedure recommended for infected tooth {}.",
    "Patient has abscess and requires root canal treatment on tooth {}."
]

cleaning_templates = [
    "Patient has plaque buildup and needs dental cleaning.",
    "Bleeding gums observed during cleaning consultation.",
    "Dental cleaning recommended because of tartar accumulation.",
    "Patient requires deep cleaning because of gum inflammation."
]

consultation_templates = [
    "Patient reports jaw pain and needs dental consultation.",
    "Consultation scheduled for tooth sensitivity and discomfort.",
    "Patient complains about oral pain and requests evaluation.",
    "Dental consultation required for ongoing mouth discomfort."
]

dataset = []

for _ in range(200):

    tooth = random.randint(1, 32)

    dataset.append({
        "text": random.choice(filling_templates).format(tooth),
        "label": "filling_form"
    })

    dataset.append({
        "text": random.choice(extraction_templates).format(tooth),
        "label": "extraction_form"
    })

    dataset.append({
        "text": random.choice(root_canal_templates).format(tooth),
        "label": "root_canal_form"
    })

    dataset.append({
        "text": random.choice(cleaning_templates),
        "label": "cleaning_form"
    })

    dataset.append({
        "text": random.choice(consultation_templates),
        "label": "consultation_form"
    })

large_df = pd.DataFrame(dataset)

large_df = large_df.sample(frac=1).reset_index(drop=True)

large_df.to_csv("iva_large_dental_dataset.csv", index=False)

print(large_df.shape)

large_df.head()

(1000, 2)


,text,label
0,Swelling and pain indicate extraction is requi...,extraction_form
1,Severe nerve pain indicates root canal is need...,root_canal_form
2,Dental examination showed cavity requiring fil...,filling_form
3,Patient reports sensitivity and filling is req...,filling_form
4,Tooth 2 extraction is needed due to severe decay.,extraction_form


In [32]:
df = pd.read_csv("iva_large_dental_dataset.csv")

df["label"].value_counts()

,count
label,
extraction_form,200
root_canal_form,200
filling_form,200
cleaning_form,200
consultation_form,200


In [33]:
labels = sorted(df["label"].unique())

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

df["label_id"] = df["label"].map(label2id)

print(label2id)

{'cleaning_form': 0, 'consultation_form': 1, 'extraction_form': 2, 'filling_form': 3, 'root_canal_form': 4}


In [34]:
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label_id"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label_id"],
    random_state=42
)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df)
})

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_id', '__index_level_0__'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['text', 'label', 'label_id', '__index_level_0__'],
        num_rows: 100
    })
    test: Dataset({
        features: ['text', 'label', 'label_id', '__index_level_0__'],
        num_rows: 100
    })
})

In [35]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)

tokenized_dataset = tokenized_dataset.rename_column("label_id", "labels")

tokenized_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [36]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [37]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.690194,0.068292,1.000000,1.000000
2,0.043666,0.019578,1.000000,1.000000
3,0.022382,0.015262,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=300, training_loss=0.25208083311716717, metrics={'train_runtime': 1675.2779, 'train_samples_per_second': 1.433, 'train_steps_per_second': 0.179, 'total_flos': 79484691456000.0, 'train_loss': 0.25208083311716717, 'epoch': 3.0})

In [38]:
results = trainer.evaluate(tokenized_dataset["test"])
results

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.015267711132764816,
 'eval_accuracy': 1.0,
 'eval_f1': 1.0,
 'eval_runtime': 19.7386,
 'eval_samples_per_second': 5.066,
 'eval_steps_per_second': 0.659,
 'epoch': 3.0}

In [39]:
predict_form("Patient requires root canal treatment because of severe infection.")

{'form_type': 'root_canal_form', 'confidence': 0.967}

In [40]:
predict_form("Patient has cavity on tooth 15 and needs composite filling.")
predict_form("Tooth 18 extraction is needed due to swelling and decay.")
predict_form("Patient requires root canal treatment because of severe infection.")
predict_form("Patient has plaque buildup and bleeding gums, cleaning required.")
predict_form("Patient reports jaw pain and needs dental consultation.")

{'form_type': 'consultation_form', 'confidence': 0.988}

In [41]:
autofill_form(
    text="Patient requires root canal treatment because of severe infection.",
    diagnosis="infection",
    procedure="root canal treatment",
    medication=None,
    tooth_number="12"
)

{'form_type': 'root_canal_form',
 'confidence': 0.967,
 'diagnosis': 'infection',
 'procedure': 'root canal treatment',
 'medication': None,
 'tooth_number': '12',
 'original_text': 'Patient requires root canal treatment because of severe infection.',
 'status': 'ready_for_review'}